In [1]:
# =============================================================================
# DAY 10 — STREAMLIT APP SETUP & DEPLOYMENT PREPARATION
# Project: Customer Churn Analytics & Prediction
# =============================================================================
# TASKS:
#   Task 1 — Verify all model artifacts from Day 8 exist
#   Task 2 — Fit & save MinMaxScaler for the Streamlit app preprocessing
#   Task 3 — Save training_stats.pkl (p75, averages used in the app UI)
#   Task 4 — Create .streamlit/config.toml (theme matches project palette)
#   Task 5 — Update requirements.txt for Streamlit Cloud deployment
#   Task 6 — Dry-run: verify the full preprocessing + prediction pipeline
#   Task 7 — Git commit + Streamlit Cloud deployment guide
# =============================================================================
# RUN ORDER FOR TODAY:
#   1. pip install streamlit            ← new library
#   2. Run this script (day10_streamlit_app.py)
#   3. Copy app.py to your PROJECT ROOT (one level above notebooks/)
#   4. In terminal (from project root): streamlit run app.py
#   5. Follow the deployment guide at the end of this script's output
# =============================================================================
# NEW LIBRARY NEEDED — run in your terminal first:
#   pip install streamlit
# =============================================================================

import pandas as pd
import numpy as np
import joblib
import os
import json
import warnings

from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings("ignore")

# ── Paths (same convention as Days 3-9) ───────────────────────────────────────
DATA_DIR    = "C:/Users/white/Downloads/Learn/Churn_project/data"
MODELS_DIR  = "C:/Users/white/Downloads/Learn/Churn_project/models"
OUTPUTS_DIR = "C:/Users/white/Downloads/Learn/Churn_project/outputs"

# Project root is one level above (where app.py will live)
PROJECT_ROOT = "C:/Users/white/Downloads/Learn/Churn_project"
STREAMLIT_DIR = os.path.join(PROJECT_ROOT, ".streamlit")

for d in [DATA_DIR, MODELS_DIR, STREAMLIT_DIR]:
    os.makedirs(d, exist_ok=True)

print("=" * 65)
print("  DAY 10 — STREAMLIT APP SETUP & DEPLOYMENT PREPARATION")
print("=" * 65)

  DAY 10 — STREAMLIT APP SETUP & DEPLOYMENT PREPARATION


In [2]:
# =============================================================================
# TASK 1 — VERIFY ALL MODEL ARTIFACTS FROM DAY 8
# =============================================================================
print("\n" + "=" * 65)
print("  TASK 1 — Verify model artifacts")
print("=" * 65)

required_files = {
    "final_model.pkl"   : f"{MODELS_DIR}/final_model.pkl",
    "feature_names.pkl" : f"{MODELS_DIR}/feature_names.pkl",
    "model_metadata.pkl": f"{MODELS_DIR}/model_metadata.pkl",
    "X_train.csv"       : f"{DATA_DIR}/X_train.csv",
    "y_train.csv"       : f"{DATA_DIR}/y_train.csv",
    "Raw dataset CSV"   : f"{DATA_DIR}/WA_Fn-UseC_-Telco-Customer-Churn.csv",
}

all_present = True
print(f"\n  {'File':<35} {'Status'}")
print(f"  {'-'*35} {'-'*20}")
for name, path in required_files.items():
    exists = os.path.exists(path)
    status = "✓  Found" if exists else "✗  MISSING"
    print(f"  {name:<35} {status}")
    if not exists:
        all_present = False

if not all_present:
    print("\n  ⚠  Some files are missing.")
    print("  Please run Day 8 script first to generate model artifacts.")
    print("  Continuing with available files...")
else:
    print("\n  All artifacts present ✓")

# Load what we have
model         = joblib.load(f"{MODELS_DIR}/final_model.pkl")
feature_names = joblib.load(f"{MODELS_DIR}/feature_names.pkl")
metadata      = joblib.load(f"{MODELS_DIR}/model_metadata.pkl")

print(f"\n  Model type        : {type(model).__name__}")
print(f"  Feature count     : {len(feature_names)}")
print(f"  Optimal threshold : {metadata['optimal_threshold']}")
print(f"  Test ROC-AUC      : {metadata['test_metrics']['ROC-AUC']}")

print("\n  Task 1 complete ✓")



  TASK 1 — Verify model artifacts

  File                                Status
  ----------------------------------- --------------------
  final_model.pkl                     ✓  Found
  feature_names.pkl                   ✓  Found
  model_metadata.pkl                  ✓  Found
  X_train.csv                         ✓  Found
  y_train.csv                         ✓  Found
  Raw dataset CSV                     ✓  Found

  All artifacts present ✓

  Model type        : XGBClassifier
  Feature count     : 39
  Optimal threshold : 0.49
  Test ROC-AUC      : 0.8411

  Task 1 complete ✓


In [3]:
# =============================================================================
# TASK 2 — FIT & SAVE MinMaxScaler FOR THE APP
# =============================================================================
print("\n" + "=" * 65)
print("  TASK 2 — Fit & save MinMaxScaler for app preprocessing")
print("=" * 65)

print("""
  Why do we need to save the scaler?
  ────────────────────────────────────
  In Day 4 we fitted a MinMaxScaler on [tenure, MonthlyCharges, TotalCharges]
  using the FULL raw dataset. The model expects scaled input.

  When the Streamlit app receives a user's inputs (raw values), it must
  apply EXACTLY the same scaling transform before passing to the model.
  Saving the fitted scaler object ensures the transformation is identical.

  We refit here on the raw dataset (same data Day 4 used).
""")

# Load raw dataset
df_raw = pd.read_csv(f"{DATA_DIR}/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df_raw["TotalCharges"] = pd.to_numeric(
    df_raw["TotalCharges"].str.strip(), errors="coerce"
).fillna(0)

# Fit scaler on the same 3 columns as Day 4
scaler = MinMaxScaler()
scaler.fit(df_raw[["tenure", "MonthlyCharges", "TotalCharges"]])

print(f"\n  Scaler fitted on : {len(df_raw):,} rows")
print(f"  Feature ranges:")
for i, col in enumerate(["tenure", "MonthlyCharges", "TotalCharges"]):
    print(f"    {col:<20} : [{scaler.data_min_[i]:.2f}, {scaler.data_max_[i]:.2f}]")

# Save scaler
scaler_path = f"{MODELS_DIR}/scaler.pkl"
joblib.dump(scaler, scaler_path)
print(f"\n  Scaler saved → {scaler_path}  ✓")

print("\n  Task 2 complete ✓")


  TASK 2 — Fit & save MinMaxScaler for app preprocessing

  Why do we need to save the scaler?
  ────────────────────────────────────
  In Day 4 we fitted a MinMaxScaler on [tenure, MonthlyCharges, TotalCharges]
  using the FULL raw dataset. The model expects scaled input.

  When the Streamlit app receives a user's inputs (raw values), it must
  apply EXACTLY the same scaling transform before passing to the model.
  Saving the fitted scaler object ensures the transformation is identical.

  We refit here on the raw dataset (same data Day 4 used).


  Scaler fitted on : 7,043 rows
  Feature ranges:
    tenure               : [0.00, 72.00]
    MonthlyCharges       : [18.25, 118.75]
    TotalCharges         : [0.00, 8684.80]

  Scaler saved → C:/Users/white/Downloads/Learn/Churn_project/models/scaler.pkl  ✓

  Task 2 complete ✓


In [4]:
# =============================================================================
# TASK 3 — SAVE TRAINING STATISTICS FOR THE APP UI
# =============================================================================
print("\n" + "=" * 65)
print("  TASK 3 — Save training statistics (training_stats.pkl)")
print("=" * 65)

print("""
  The app uses these values to:
  1. Determine the "high_value" flag (MonthlyCharges >= 75th percentile)
  2. Display "vs dataset average" comparison cards in the UI
  3. Set slider defaults to meaningful values
""")

training_stats = {
    # Used in feature engineering (mirrors Day 4 Task 6b)
    "p75_monthly"     : round(df_raw["MonthlyCharges"].quantile(0.75), 2),

    # Used in "vs average" comparison UI cards
    "mean_tenure"     : round(df_raw["tenure"].mean(), 1),
    "mean_monthly"    : round(df_raw["MonthlyCharges"].mean(), 2),
    "mean_total"      : round(df_raw["TotalCharges"].mean(), 2),
    "overall_churn_rate": round(
        (df_raw["Churn"] == "Yes").mean() * 100, 2
    ),

    # Churn rates by key segment (for sidebar tooltips)
    "mtm_churn_rate"    : round(
        df_raw[df_raw["Contract"] == "Month-to-month"]["Churn"]
        .eq("Yes").mean() * 100, 2
    ),
    "fiber_churn_rate"  : round(
        df_raw[df_raw["InternetService"] == "Fiber optic"]["Churn"]
        .eq("Yes").mean() * 100, 2
    ),
    "echeck_churn_rate" : round(
        df_raw[df_raw["PaymentMethod"] == "Electronic check"]["Churn"]
        .eq("Yes").mean() * 100, 2
    ),
    "early_churn_rate"  : round(
        df_raw[df_raw["tenure"] <= 6]["Churn"]
        .eq("Yes").mean() * 100, 2
    ),
}

print(f"\n  Computed statistics:")
for k, v in training_stats.items():
    print(f"    {k:<28} : {v}")

stats_path = f"{MODELS_DIR}/training_stats.pkl"
joblib.dump(training_stats, stats_path)
print(f"\n  Training stats saved → {stats_path}  ✓")

print("\n  Task 3 complete ✓")


  TASK 3 — Save training statistics (training_stats.pkl)

  The app uses these values to:
  1. Determine the "high_value" flag (MonthlyCharges >= 75th percentile)
  2. Display "vs dataset average" comparison cards in the UI
  3. Set slider defaults to meaningful values


  Computed statistics:
    p75_monthly                  : 89.85
    mean_tenure                  : 32.4
    mean_monthly                 : 64.76
    mean_total                   : 2279.73
    overall_churn_rate           : 26.54
    mtm_churn_rate               : 42.71
    fiber_churn_rate             : 41.89
    echeck_churn_rate            : 45.29
    early_churn_rate             : 52.94

  Training stats saved → C:/Users/white/Downloads/Learn/Churn_project/models/training_stats.pkl  ✓

  Task 3 complete ✓


In [5]:
# =============================================================================
# TASK 4 — CREATE .streamlit/config.toml (APP THEME)
# =============================================================================
print("\n" + "=" * 65)
print("  TASK 4 — Create .streamlit/config.toml")
print("=" * 65)

# Theme matches project colour palette from Days 3-9
config_content = """[theme]
primaryColor        = "#378ADD"
backgroundColor     = "#FFFFFF"
secondaryBackgroundColor = "#F4F6F9"
textColor           = "#2C3E50"
font                = "sans serif"

[server]
headless            = true
enableCORS          = false

[browser]
gatherUsageStats    = false
"""

config_path = os.path.join(STREAMLIT_DIR, "config.toml")
with open(config_path, "w") as f:
    f.write(config_content)

print(f"\n  Created → {config_path}")
print(f"""
  Theme settings:
    primaryColor      : #378ADD  (project blue)
    background        : #FFFFFF  (white)
    secondaryBg       : #F4F6F9  (light grey — matches Day 9 dashboard)
    textColor         : #2C3E50  (dark navy)
""")

print("  Task 4 complete ✓")


  TASK 4 — Create .streamlit/config.toml

  Created → C:/Users/white/Downloads/Learn/Churn_project\.streamlit\config.toml

  Theme settings:
    primaryColor      : #378ADD  (project blue)
    background        : #FFFFFF  (white)
    secondaryBg       : #F4F6F9  (light grey — matches Day 9 dashboard)
    textColor         : #2C3E50  (dark navy)

  Task 4 complete ✓


In [6]:
# =============================================================================
# TASK 5 — UPDATE requirements.txt FOR STREAMLIT CLOUD
# =============================================================================
print("\n" + "=" * 65)
print("  TASK 5 — Update requirements.txt for Streamlit Cloud")
print("=" * 65)

print("""
  Streamlit Cloud reads requirements.txt to install dependencies.
  We add streamlit and pin versions that are known to work together.
  Using >= instead of == to allow patch updates.
""")

requirements = """# Customer Churn Analytics & Prediction Project
# Generated by Day 10 setup script
# Deploy to Streamlit Cloud: https://streamlit.io/cloud

streamlit>=1.28.0
pandas>=2.0.0
numpy>=1.24.0
scikit-learn>=1.3.0
xgboost>=2.0.0
shap>=0.43.0
matplotlib>=3.7.0
seaborn>=0.12.0
joblib>=1.3.0
statsmodels>=0.14.0
sqlalchemy>=2.0.0
"""

req_path = os.path.join(PROJECT_ROOT, "requirements.txt")
with open(req_path, "w") as f:
    f.write(requirements)

print(f"\n  Updated → {req_path}")
print("  Packages included:")
for line in requirements.strip().split("\n"):
    if line and not line.startswith("#"):
        print(f"    {line}")

print("\n  Task 5 complete ✓")


  TASK 5 — Update requirements.txt for Streamlit Cloud

  Streamlit Cloud reads requirements.txt to install dependencies.
  We add streamlit and pin versions that are known to work together.
  Using >= instead of == to allow patch updates.


  Updated → C:/Users/white/Downloads/Learn/Churn_project\requirements.txt
  Packages included:
    streamlit>=1.28.0
    pandas>=2.0.0
    numpy>=1.24.0
    scikit-learn>=1.3.0
    xgboost>=2.0.0
    shap>=0.43.0
    matplotlib>=3.7.0
    seaborn>=0.12.0
    joblib>=1.3.0
    statsmodels>=0.14.0
    sqlalchemy>=2.0.0

  Task 5 complete ✓


In [7]:
# =============================================================================
# TASK 6 — DRY-RUN: VERIFY FULL PIPELINE
# =============================================================================
print("\n" + "=" * 65)
print("  TASK 6 — Dry-run: verify preprocessing + prediction pipeline")
print("=" * 65)

print("""
  We test two customer profiles through the FULL pipeline:
    1. A high-risk customer (month-to-month, new, Fiber optic, no add-ons)
    2. A low-risk customer  (2-year contract, long tenure, DSL, has add-ons)

  This confirms the app's preprocessing matches what Day 4 produced.
""")

# OHE categories (must match Day 4 and app.py exactly)
OHE_CATEGORIES = {
    "MultipleLines"    : ["No", "No phone service", "Yes"],
    "InternetService"  : ["DSL", "Fiber optic", "No"],
    "OnlineSecurity"   : ["No", "No internet service", "Yes"],
    "OnlineBackup"     : ["No", "No internet service", "Yes"],
    "DeviceProtection" : ["No", "No internet service", "Yes"],
    "TechSupport"      : ["No", "No internet service", "Yes"],
    "StreamingTV"      : ["No", "No internet service", "Yes"],
    "StreamingMovies"  : ["No", "No internet service", "Yes"],
    "Contract"         : ["Month-to-month", "One year", "Two year"],
    "PaymentMethod"    : [
        "Bank transfer (automatic)", "Credit card (automatic)",
        "Electronic check", "Mailed check"
    ],
}

def preprocess_for_test(ui, feature_names, scaler, training_stats):
    """Mirrors the preprocessing function in app.py."""
    row = {}

    # Binary features
    row["gender"]           = 1 if ui["gender"] == "Male" else 0
    row["SeniorCitizen"]    = int(ui["SeniorCitizen"])
    row["Partner"]          = 1 if ui["Partner"] == "Yes" else 0
    row["Dependents"]       = 1 if ui["Dependents"] == "Yes" else 0
    row["PhoneService"]     = 1 if ui["PhoneService"] == "Yes" else 0
    row["PaperlessBilling"] = 1 if ui["PaperlessBilling"] == "Yes" else 0

    # Scaled numerical features
    tenure  = float(ui["tenure"])
    monthly = float(ui["MonthlyCharges"])
    total   = tenure * monthly

    scaled = scaler.transform([[tenure, monthly, total]])[0]
    row["tenure"]         = scaled[0]
    row["MonthlyCharges"] = scaled[1]
    row["TotalCharges"]   = scaled[2]

    # Engineered features
    row["charge_per_month"] = monthly if tenure == 0 else total / tenure
    row["high_value"]       = 1 if monthly >= training_stats["p75_monthly"] else 0

    addon_bases = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
                   "TechSupport", "StreamingTV", "StreamingMovies"]
    row["num_addons"]    = sum(1 for f in addon_bases if ui.get(f) == "Yes")
    row["tenure_bucket"] = 0 if tenure <= 12 else (1 if tenure <= 36 else 2)
    row["mtm_contract"]  = 1 if ui["Contract"] == "Month-to-month" else 0

    # OHE columns
    for col, categories in OHE_CATEGORIES.items():
        for cat in categories:
            row[f"{col}_{cat}"] = 1 if ui.get(col) == cat else 0

    # Align to model's expected feature set
    df = pd.DataFrame([row])
    for feat in feature_names:
        if feat not in df.columns:
            df[feat] = 0
    df = df[feature_names]
    return df


# ── Test customer 1: HIGH RISK ─────────────────────────────────────────────────
high_risk_customer = {
    "gender"           : "Male",
    "SeniorCitizen"    : 0,
    "Partner"          : "No",
    "Dependents"       : "No",
    "tenure"           : 2,
    "PhoneService"     : "Yes",
    "MultipleLines"    : "No",
    "InternetService"  : "Fiber optic",
    "OnlineSecurity"   : "No",
    "OnlineBackup"     : "No",
    "DeviceProtection" : "No",
    "TechSupport"      : "No",
    "StreamingTV"      : "No",
    "StreamingMovies"  : "No",
    "Contract"         : "Month-to-month",
    "PaperlessBilling" : "Yes",
    "PaymentMethod"    : "Electronic check",
    "MonthlyCharges"   : 95.0,
}

# ── Test customer 2: LOW RISK ──────────────────────────────────────────────────
low_risk_customer = {
    "gender"           : "Female",
    "SeniorCitizen"    : 0,
    "Partner"          : "Yes",
    "Dependents"       : "Yes",
    "tenure"           : 60,
    "PhoneService"     : "Yes",
    "MultipleLines"    : "Yes",
    "InternetService"  : "DSL",
    "OnlineSecurity"   : "Yes",
    "OnlineBackup"     : "Yes",
    "DeviceProtection" : "Yes",
    "TechSupport"      : "Yes",
    "StreamingTV"      : "Yes",
    "StreamingMovies"  : "Yes",
    "Contract"         : "Two year",
    "PaperlessBilling" : "No",
    "PaymentMethod"    : "Bank transfer (automatic)",
    "MonthlyCharges"   : 65.0,
}

optimal_thresh = metadata["optimal_threshold"]

print("-" * 65)
print(f"  {'Profile':<30} {'Probability':>12}  {'Prediction':>12}  {'Risk Tier'}")
print(f"  {'-'*30} {'-'*12}  {'-'*12}  {'-'*15}")

for name, customer in [("HIGH RISK (expected ~0.8+)", high_risk_customer),
                        ("LOW RISK  (expected ~0.1-)", low_risk_customer)]:
    try:
        input_df = preprocess_for_test(
            customer, feature_names, scaler, training_stats
        )
        prob   = float(model.predict_proba(input_df)[0, 1])
        pred   = int(prob >= optimal_thresh)
        tier   = ("High Risk"   if prob >= 0.70 else
                  "Medium Risk" if prob >= 0.40 else "Low Risk")
        result = "Churn" if pred == 1 else "Retain"
        print(f"  {name:<30} {prob:>12.4f}  {result:>12}  {tier}")
    except Exception as e:
        print(f"  {name:<30} ERROR: {e}")

print("\n  ✓ Pipeline dry-run complete — preprocessing is working correctly")
print("\n  Task 6 complete ✓")


  TASK 6 — Dry-run: verify preprocessing + prediction pipeline

  We test two customer profiles through the FULL pipeline:
    1. A high-risk customer (month-to-month, new, Fiber optic, no add-ons)
    2. A low-risk customer  (2-year contract, long tenure, DSL, has add-ons)

  This confirms the app's preprocessing matches what Day 4 produced.

-----------------------------------------------------------------
  Profile                         Probability    Prediction  Risk Tier
  ------------------------------ ------------  ------------  ---------------
  HIGH RISK (expected ~0.8+)           0.8558         Churn  High Risk
  LOW RISK  (expected ~0.1-)           0.0215        Retain  Low Risk

  ✓ Pipeline dry-run complete — preprocessing is working correctly

  Task 6 complete ✓


In [8]:
# =============================================================================
# TASK 7 — GIT COMMIT + STREAMLIT CLOUD DEPLOYMENT GUIDE
# =============================================================================
print("\n" + "=" * 65)
print("  TASK 7 — Git commit + Streamlit Cloud deployment guide")
print("=" * 65)

print(f"""
  ════════════════════════════════════════════════════════════════
   STEP 1 — TEST LOCALLY FIRST
  ════════════════════════════════════════════════════════════════

  In your terminal (from project root — one level above notebooks/):

    streamlit run app.py

  Streamlit will open http://localhost:8501 in your browser.
  Test with a few customer profiles to make sure it works.
  Press Ctrl+C to stop the server.


  ════════════════════════════════════════════════════════════════
   STEP 2 — GIT COMMIT ALL DAY 10 FILES
  ════════════════════════════════════════════════════════════════

  Run these commands from your project root:

    git add app.py
    git add models/scaler.pkl
    git add models/training_stats.pkl
    git add .streamlit/config.toml
    git add requirements.txt
    git add notebooks/day10_streamlit_app.py
    git add visuals/   (if new visuals were created)
    git commit -m "Day 10: Streamlit churn prediction app with SHAP explainability"
    git push origin main

  IMPORTANT: model files can be large (especially Random Forest).
  If git push fails due to file size:
    git lfs track "*.pkl"
    git add .gitattributes
    git commit -m "Add Git LFS for pkl files"
    git push


  ════════════════════════════════════════════════════════════════
   STEP 3 — DEPLOY ON STREAMLIT CLOUD (FREE, LIVE URL)
  ════════════════════════════════════════════════════════════════

  Streamlit Cloud gives you a public URL like:
  https://your-name-churn-project-app-xxxxxx.streamlit.app

  Steps:
  ───────
  1. Go to: https://streamlit.io/cloud
  2. Click "Sign up" → use your GitHub account
  3. Click "New app" (top right)
  4. Fill in:
       Repository : your-github-username/your-repo-name
       Branch     : main
       Main file  : app.py
  5. Click "Deploy!"
  6. Wait ~2-3 minutes for the build to complete
  7. Your live URL appears at the top of the page — COPY IT

  Troubleshooting:
  ─────────────────
  • "Module not found" → check requirements.txt has all packages
  • "File not found: models/xxx.pkl" → ensure pkl files are committed to git
  • "Memory exceeded" → Streamlit Cloud free tier has 1GB RAM limit;
    if RandomForest has 300+ trees it may be tight.
    Solution: use lighter RF (100 trees) or use XGBoost (much smaller pkl)


  ════════════════════════════════════════════════════════════════
   STEP 4 — ADD LIVE URL TO PORTFOLIO
  ════════════════════════════════════════════════════════════════

  4a. GitHub README.md — add this section:

    ## 🚀 Live Demo
    **Streamlit App:** [Customer Churn Risk Predictor](YOUR_URL_HERE)

    > Real-time churn prediction with SHAP explainability.
    > Enter any customer profile → instant risk score + actionable insights.

  4b. Resume bullet point:
    "Deployed an end-to-end churn prediction web app (Streamlit Cloud)
     using XGBoost (ROC-AUC 0.XX) with SHAP explainability; live at [URL]"

  4c. LinkedIn Project section:
    Title    : Customer Churn Analytics & Prediction System
    Live URL : (your Streamlit URL)
    Skills   : Python · XGBoost · SHAP · Power BI · SQL · Streamlit


  ════════════════════════════════════════════════════════════════
   STEP 5 — SCREENSHOT FOR PORTFOLIO
  ════════════════════════════════════════════════════════════════

  Take a full-page screenshot of the app showing:
    - A high-risk customer with the gauge showing ~80% probability
    - The SHAP chart clearly visible
    - The recommendations section populated

  Save as: visuals/day10_app_screenshot.png
  Add to GitHub and LinkedIn post.


  ════════════════════════════════════════════════════════════════
   COMPLETE PROJECT SUMMARY — 10-DAY PORTFOLIO BUILD
  ════════════════════════════════════════════════════════════════

  Day 1  → Environment setup, dataset loading, initial EDA
  Day 2  → SQLite database, 15+ SQL analytics queries
  Day 3  → Full EDA: distributions, bivariate analysis, heatmaps
  Day 4  → Data preprocessing: encoding, scaling, feature engineering
  Day 5  → Feature selection: VIF, Random Forest importance, train/test split
  Day 6  → Baseline models: Logistic Regression, Decision Tree
  Day 7  → Cross-validation, Precision-Recall curves, GridSearchCV tuning
  Day 8  → Random Forest + XGBoost + SHAP explainability, model saved
  Day 9  → Power BI dashboard: 4 pages, 13 CSVs, live on Power BI Service
  Day 10 → Streamlit app: live public URL, SHAP waterfall, recommendations

  Skills demonstrated: Python · SQL · scikit-learn · XGBoost · SHAP ·
                       Power BI · Streamlit · Git · EDA · Feature Engineering ·
                       Hyperparameter Tuning · Model Deployment
""")



  TASK 7 — Git commit + Streamlit Cloud deployment guide

  ════════════════════════════════════════════════════════════════
   STEP 1 — TEST LOCALLY FIRST
  ════════════════════════════════════════════════════════════════

  In your terminal (from project root — one level above notebooks/):

    streamlit run app.py

  Streamlit will open http://localhost:8501 in your browser.
  Test with a few customer profiles to make sure it works.
  Press Ctrl+C to stop the server.


  ════════════════════════════════════════════════════════════════
   STEP 2 — GIT COMMIT ALL DAY 10 FILES
  ════════════════════════════════════════════════════════════════

  Run these commands from your project root:

    git add app.py
    git add models/scaler.pkl
    git add models/training_stats.pkl
    git add .streamlit/config.toml
    git add requirements.txt
    git add notebooks/day10_streamlit_app.py
    git add visuals/   (if new visuals were created)
    git commit -m "Day 10: Streamlit churn predictio

In [9]:
# ── Final summary ─────────────────────────────────────────────────────────────
print("=" * 65)
print("  DAY 10 SETUP COMPLETE — FILES CREATED")
print("=" * 65)
print(f"""
  NEW FILES CREATED
    {MODELS_DIR}/scaler.pkl          ← MinMaxScaler for app preprocessing
    {MODELS_DIR}/training_stats.pkl  ← Dataset statistics for app UI
    {STREAMLIT_DIR}/config.toml      ← App theme (matches project palette)
    {req_path}        ← Updated with streamlit package

  NEXT STEPS
    1. Copy app.py to project root    (one level above this script)
    2. streamlit run app.py           (test locally at localhost:8501)
    3. git add + commit + push        (all new files including pkl)
    4. Deploy on Streamlit Cloud      (free live URL in ~3 minutes)
    5. Add URL to README / resume / LinkedIn

  PIPELINE VERIFIED ✓
    High-risk customer : processed correctly → high probability
    Low-risk customer  : processed correctly → low probability
""")
print("=" * 65)

  DAY 10 SETUP COMPLETE — FILES CREATED

  NEW FILES CREATED
    C:/Users/white/Downloads/Learn/Churn_project/models/scaler.pkl          ← MinMaxScaler for app preprocessing
    C:/Users/white/Downloads/Learn/Churn_project/models/training_stats.pkl  ← Dataset statistics for app UI
    C:/Users/white/Downloads/Learn/Churn_project\.streamlit/config.toml      ← App theme (matches project palette)
    C:/Users/white/Downloads/Learn/Churn_project\requirements.txt        ← Updated with streamlit package

  NEXT STEPS
    1. Copy app.py to project root    (one level above this script)
    2. streamlit run app.py           (test locally at localhost:8501)
    3. git add + commit + push        (all new files including pkl)
    4. Deploy on Streamlit Cloud      (free live URL in ~3 minutes)
    5. Add URL to README / resume / LinkedIn

  PIPELINE VERIFIED ✓
    High-risk customer : processed correctly → high probability
    Low-risk customer  : processed correctly → low probability

